# 01 — Data Collection & Exploration
Downloads exchange rate data and performs initial EDA.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from ml.data_pipeline import download_all, load_all, summarize, run_pipeline

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Download Data

In [ ]:
# Downloads and saves CSVs to data/raw/
raw_data = download_all()
summarize(raw_data)

## 2. Plot Close Prices

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)

for ax, (name, df) in zip(axes, raw_data.items()):
    ax.plot(df['Date'], df['Close'], linewidth=1)
    ax.set_title(f'{name.replace("_", "/")} — Daily Close Price')
    ax.set_ylabel('Exchange Rate')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('../data/processed/close_prices.png', dpi=150)
plt.show()

## 3. Run Preprocessing Pipeline

In [ ]:
processed = {}
for name, df in raw_data.items():
    processed[name] = run_pipeline(df, pair_name=name, sequence_length=60)

print('Preprocessing complete.')
for name, p in processed.items():
    print(f'  {name}: X_train={p["X_train"].shape}, X_val={p["X_val"].shape}, X_test={p["X_test"].shape}')

## 4. Log Returns Distribution

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, df) in zip(axes, raw_data.items()):
    log_ret = np.log(df['Close'] / df['Close'].shift(1)).dropna()
    ax.hist(log_ret, bins=80, edgecolor='none', alpha=0.8)
    ax.set_title(f'{name.replace("_", "/")} — Log Returns')
    ax.set_xlabel('Log Return')
plt.tight_layout()
plt.show()

## 5. Correlation Between Pairs

In [ ]:
closes = pd.DataFrame({
    name: df.set_index('Date')['Close']
    for name, df in raw_data.items()
}).dropna()

log_returns = np.log(closes / closes.shift(1)).dropna()
corr = log_returns.corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Log Return Correlation Matrix')
plt.tight_layout()
plt.show()
print(corr)